In [1]:
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
from typing import TypedDict,Annotated
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel,Field
import operator




In [2]:
load_dotenv()




True

In [3]:
model = ChatGoogleGenerativeAI(model="gemini 2.5 Flash")

In [5]:
class EvaluateSchema(BaseModel):
    
    feedback:str=Field(description='Detailed feedback for the essay')
    score: int = Field(description='Score out of 10',ge=0,le=10)

In [8]:
structured_model=model.with_structured_output(EvaluateSchema)


In [11]:
class EssayState(TypedDict):

    essay:str
    language_feedback:str
    clarity_feedback:str
    overall_feedback:str
    individual_scores:Annotated[list[int],operator.add] #add reducer function
    avg_score:float

In [14]:
#functions

def evaluate_language(state:EssayState):
    prompt = f"Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10:\n{state['essay']}"
    output=structured_model.invoke(prompt)
    return {'language_feedback':output.feedback,'individual_scores':[output.score]}


In [16]:
def evaluate_analysis(state:EssayState):
    prompt = f"Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10:\n{state['essay']}"
    output=structured_model.invoke(prompt)
    return {'analysis_feedback':output.feedback,'individual_scores':[output.score]}

In [18]:
def evaluate_thought(state:EssayState):
    prompt = f"Evaluate the clarity of thought  of the following essay and provide a feedback and assign a score out of 10:\n{state['essay']}"
    output=structured_model.invoke(prompt)
    return {'clarity_feedback':output.feedback,'individual_scores':[output.score]}

In [ ]:
def final_evaluation(state: EssayState):

    #sumary feedback
    prompt = f'prompt = f"Based on the following feedback, create a summarized feedback \n  language feedback- {state["language_feedback"]} \n depth of analysis feedback - {state["analysis_feedback"]} \n  clarity of thought feedback - {state["clarity_feedback"]}"'

    overall_feedback=model.invoke(prompt).content
    
    #avg-calculate
    avg_score=sum(state['individual_scores'])/len(state['individual_scores'])

    return {'overall_feedback':overall_feedback,'avg_score':avg_score}




    

In [ ]:
#graph
graph=StateGraph(EssayState)

#nodes
# Add all evaluation nodes to the graph
graph.add_node("evaluate_language", evaluate_language)
graph.add_node("evaluate_analysis", evaluate_analysis)
graph.add_node("evaluate_thought", evaluate_thought)
graph.add_node("evaluate_clarity", evaluate_clarity)
graph.add_node("final_evaluation", final_evaluation)




NameError: name 'graph_builder' is not defined